# import_

> Import route for file upload with JSON validation and merge strategy

In [ ]:
#| default_exp routes.import_

In [ ]:
#| export
import json
from typing import Dict, Callable, Tuple

from fasthtml.common import APIRouter, UploadFile, Div

from cjm_transcript_workflow_management.services.management import ManagementService
from cjm_transcript_workflow_management.models import ManagementUrls, ImportResult
from cjm_transcript_workflow_management.html_ids import ManagementHtmlIds
from cjm_transcript_workflow_management.components.import_controls import render_import_result
from cjm_transcript_workflow_management.components.document_list import render_document_list
from cjm_transcript_workflow_management.routes.core import DEBUG_MANAGEMENT_ROUTES

## Router Initialization

In [ ]:
#| export
def init_import_router(
    service:ManagementService,  # Service for graph queries
    prefix:str,  # Route prefix (e.g., "/manage/import")
    urls:ManagementUrls,  # URL bundle (for list refresh)
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, routes dict)
    """Initialize import route for file upload with merge strategy."""
    router = APIRouter(prefix=prefix)
    routes = {}
    
    @router.post
    async def import_graph(file:UploadFile, merge_strategy:str="skip"):
        """Import a graph from an uploaded JSON file."""
        if DEBUG_MANAGEMENT_ROUTES:
            print(f"[ROUTES] import_graph called, strategy={merge_strategy}")
        
        # Validate file was provided
        if file is None or not file.filename:
            result = ImportResult(
                success=False, nodes_created=0, edges_created=0,
                nodes_skipped=0, edges_skipped=0,
                errors=["No file provided"],
            )
            return render_import_result(result)
        
        # Read and parse JSON
        try:
            content = await file.read()
            bundle_data = json.loads(content)
        except json.JSONDecodeError as e:
            result = ImportResult(
                success=False, nodes_created=0, edges_created=0,
                nodes_skipped=0, edges_skipped=0,
                errors=[f"Invalid JSON: {e}"],
            )
            return render_import_result(result)
        except Exception as e:
            result = ImportResult(
                success=False, nodes_created=0, edges_created=0,
                nodes_skipped=0, edges_skipped=0,
                errors=[f"Error reading file: {e}"],
            )
            return render_import_result(result)
        
        if DEBUG_MANAGEMENT_ROUTES:
            print(f"[ROUTES] Parsed JSON, format={bundle_data.get('format')}, "
                  f"version={bundle_data.get('version')}")
        
        # Import via service (handles format/version validation)
        result = await service.import_graph_async(bundle_data, merge_strategy)
        
        if DEBUG_MANAGEMENT_ROUTES:
            print(f"[ROUTES] Import result: success={result.success}, "
                  f"nodes={result.nodes_created}, edges={result.edges_created}")
        
        # Return result alert + OOB refresh of document list
        # Note: must use hyphenated key on .attrs dict (underscore keys are NOT converted)
        documents = await service.list_documents_async()
        list_html = render_document_list(documents, urls)
        list_html.attrs['hx-swap-oob'] = 'true'
        
        return render_import_result(result), list_html
    
    routes["import_graph"] = import_graph
    
    return router, routes

In [ ]:
assert init_import_router is not None
print("routes.import_ imports OK")

routes.import_ imports OK


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()